In [10]:
# =============================================================================
# FASE 1: MAPA ACÚSTICO + CURVA DE OURO PONDERADA + MOTOR DE REVERB
# =============================================================================
import os
import ast
import json
import random
import certifi
import numpy as np
import pandas as pd
import librosa
from datasets import load_dataset
from tqdm import tqdm

os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("=====================================================================")
print(" FASE 1: CURADORIA SOCIALFX (EQ + REVERB) E MAPEAMENTO EMPÍRICO")
print("=====================================================================")

# --- Parâmetros Globais ---
LIMIAR_CONSISTENCIA_EQ = 0.500
MIN_PALAVRAS_AGREED_REVERB = 1
MAX_VIRGULAS_TEXT_REVERB = 2
TOP_K_POR_SPLIT = 300  # <- Restaurado!
NOTAS_POR_FAMILIA = 60
PARES_POR_NOTA = 2     # <- Mantido para travar a explosão do dataset

freqs_eq = np.logspace(np.log10(20), np.log10(20000), num=40)

# Dicionário Expandido de Pares Contrastivos (Antônimos Perfeitos)
PARES_OPOSTOS_OURO = {
    "bright": "dark", "dark": "bright",
    "warm": "tinny", "tinny": "warm",
    "muffled": "crisp", "crisp": "muffled",
    "harsh": "soft", "soft": "harsh",
    "metallic": "mellow", "mellow": "metallic",
    "deep": "thin", "thin": "deep",
    "bassy": "trebly", "trebly": "bassy",
    # Adicionando pares espaciais nativos do SocialFX
    "spacious": "dry", "dry": "spacious",
    "distant": "close", "close": "distant"
}

TERMOS_INVERTIDOS_PELO_SOCIALFX = {"bright", "dark", "tinny", "harsh", "metallic", "reed"}
TERMOS_ESTRITAMENTE_ESPACIAIS = {"spacious", "echo", "dry", "distant", "close"}

# --- Funções Auxiliares ---
def parse_extra_data(extra_str):
    if isinstance(extra_str, str):
        try: return ast.literal_eval(extra_str)
        except: return {}
    return extra_str if isinstance(extra_str, dict) else {}

def parse_params(param_val):
    if isinstance(param_val, str):
        try: return ast.literal_eval(param_val)
        except: return []
    return param_val if param_val is not None else []

def calc_consensus_strength_reverb(extra_dict):
    agreed_raw = extra_dict.get('agreed', '')
    if not agreed_raw: return 0
    tokens_validos = [w.strip() for w in agreed_raw.split(',') if w.strip() and w.strip().lower() not in ('none', 'none of the above', '')]
    return len(tokens_validos)

# --- 1. Carregamento e Processamento do SocialFX ---
print("Baixando SocialFX (EQ e Reverb) do Hugging Face...")
dataset_eq = load_dataset("seungheondoh/socialfx-original", split="eq").to_pandas()
dataset_reverb = load_dataset("seungheondoh/socialfx-original", split="reverb").to_pandas()

# Tratamento EQ (Com Média Ponderada de Ouro!)
dataset_eq['extra_dict'] = dataset_eq['extra'].apply(parse_extra_data)
dataset_eq['lang'] = dataset_eq['extra_dict'].apply(lambda x: x.get('lang', ''))
dataset_eq['consistency'] = dataset_eq['extra_dict'].apply(lambda x: x.get('ratings_consistency', 0))
dataset_eq['param_values_list'] = dataset_eq['param_values'].apply(parse_params)

BLACKLIST_EQ = ['taco', 'baseball', 'disgusting', 'genius', 'poor']
df_eq_filtrado = dataset_eq[(dataset_eq['lang'] == 'English') & (dataset_eq['consistency'] >= LIMIAR_CONSISTENCIA_EQ) & (~dataset_eq['text'].isin(BLACKLIST_EQ))].copy()

# O FUNIL RESTAURADO PARA O EQ: Ordena por consistência e pega os TOP_K
df_eq_filtrado = df_eq_filtrado.sort_values('consistency', ascending=False).drop_duplicates(subset=['text'], keep='first')
df_eq_ouro = df_eq_filtrado.head(TOP_K_POR_SPLIT).copy()

# Tratamento Reverb
dataset_reverb['extra_dict'] = dataset_reverb['extra'].apply(parse_extra_data)
dataset_reverb['lang'] = dataset_reverb['extra_dict'].apply(lambda x: x.get('lang', ''))
dataset_reverb['consensus_strength'] = dataset_reverb['extra_dict'].apply(calc_consensus_strength_reverb)

df_reverb_filtrado = dataset_reverb[(dataset_reverb['lang'] == 'English') & (dataset_reverb['consensus_strength'] >= MIN_PALAVRAS_AGREED_REVERB) & (dataset_reverb['text'].str.count(',') <= MAX_VIRGULAS_TEXT_REVERB)].copy()
df_reverb_filtrado['text'] = df_reverb_filtrado['text'].str.replace(',', ', ', regex=False)
df_reverb_filtrado = df_reverb_filtrado.sort_values('consensus_strength', ascending=False).drop_duplicates(subset=['text'], keep='first')
df_reverb_filtrado['param_values_list'] = df_reverb_filtrado['param_values'].apply(parse_params)
df_reverb_filtrado['tipo_efeito'] = 'reverb'
df_reverb_filtrado['consistencia'] = df_reverb_filtrado['consensus_strength'].astype(float)
df_reverb_ouro = df_reverb_filtrado.head(TOP_K_POR_SPLIT).copy()

# Construção do Vocabulário Unificado
vocabulario_dsp = {}

# Média Ponderada para EQ usando o df de OURO
for texto, group in df_eq_ouro.groupby('text'):
    if texto in TERMOS_ESTRITAMENTE_ESPACIAIS: continue
    
    curvas_validas = group[group['param_values_list'].apply(len) == 40]
    if len(curvas_validas) == 0: continue
        
    curvas = np.array(curvas_validas['param_values_list'].tolist())
    pesos = curvas_validas['consistency'].values
    
    curva_ouro = np.average(curvas, axis=0, weights=pesos) if np.sum(pesos) > 0 else np.mean(curvas, axis=0)
    if texto in TERMOS_INVERTIDOS_PELO_SOCIALFX: curva_ouro = -curva_ouro
        
    vocabulario_dsp[texto] = {
        'tipo': 'eq',
        'params': curva_ouro.tolist(),
        'consistencia': float(np.mean(pesos)),
        'eh_multipalavra': False,
        'bandas_ativas': np.where(np.abs(curva_ouro) > 0.2)[0].tolist()
    }

# Insere o Reverb no Vocabulário
for _, row in df_reverb_ouro.iterrows():
    texto = row['text']
    if texto not in vocabulario_dsp:
        vocabulario_dsp[texto] = {
            'tipo': 'reverb', 'params': row['param_values_list'],
            'consistencia': row['consistencia'], 'eh_multipalavra': ', ' in texto,
            'bandas_ativas': [] # Reverb não depende de bandas de frequência específicas
        }

# --- 2. Análise Espectral do NSynth (A Trava Metodológica) ---
print("\nAnalisando distribuição de energia harmônica no NSynth Valid...")
with open('nsynth-valid/examples.json', 'r') as f:
    nsynth_meta = json.load(f)

energia_instrumentos = {}
contagem_instrumentos = {}
DIR_AUDIO_DRY = "nsynth-valid/audio"

for nota_id, info in tqdm(nsynth_meta.items(), desc="Varrendo Espectros"):
    familia = info.get('instrument_family_str')
    caminho = os.path.join(DIR_AUDIO_DRY, f"{nota_id}.wav")
    
    if not os.path.exists(caminho): continue
    if familia not in energia_instrumentos:
        energia_instrumentos[familia] = np.zeros(40)
        contagem_instrumentos[familia] = 0
    if contagem_instrumentos[familia] >= 150: continue
        
    try:
        y, sr = librosa.load(caminho, sr=22050, mono=True)
        stft = np.abs(librosa.stft(y))
        fft_freqs = librosa.fft_frequencies(sr=sr)
        espectro_medio = np.mean(stft, axis=1)
        
        energia_40_bandas = np.zeros(40)
        for i, f_alvo in enumerate(freqs_eq):
            idx = np.where((fft_freqs >= f_alvo * 0.8) & (fft_freqs <= f_alvo * 1.2))[0]
            if len(idx) > 0: energia_40_bandas[i] = np.mean(espectro_medio[idx])
                
        if np.max(energia_40_bandas) > 0: energia_40_bandas /= np.max(energia_40_bandas)
        energia_instrumentos[familia] += energia_40_bandas
        contagem_instrumentos[familia] += 1
    except: continue

for fam in energia_instrumentos:
    if contagem_instrumentos[fam] > 0: energia_instrumentos[fam] /= contagem_instrumentos[fam]

# --- 3. Dicionário de Compatibilidade (Pico Harmônico > 0.25) ---
print("\nConstruindo Dicionário Dinâmico de Compatibilidade (Corte por Pico: 0.25)...")
dicionario_compatibilidade = {fam: [] for fam in energia_instrumentos.keys()}

for fam, energia_perfil in energia_instrumentos.items():
    for termo, oposto in PARES_OPOSTOS_OURO.items():
        if termo not in vocabulario_dsp or oposto not in vocabulario_dsp: continue
            
        info_a = vocabulario_dsp[termo]
        info_o = vocabulario_dsp[oposto]
        par_ordenado = sorted([termo, oposto])
        
        if par_ordenado in dicionario_compatibilidade[fam]: continue
            
        # Reverb passa direto, EQ sofre a trava de Energia por PICO
        if info_a['tipo'] == 'eq' and info_o['tipo'] == 'eq':
            bandas = set(info_a['bandas_ativas'] + info_o['bandas_ativas'])
            if not bandas: continue
                
            energia_pico_alvo = np.max([energia_perfil[b] for b in bandas])
            
            if energia_pico_alvo > 0.25: 
                dicionario_compatibilidade[fam].append(par_ordenado)
        else:
            dicionario_compatibilidade[fam].append(par_ordenado)

# --- 4. Geração do Blueprint ---
print("\nGerando Blueprint Otimizado por Compatibilidade...")
agrupamento_notas = {}
for nota_id, info in nsynth_meta.items():
    fam = info.get('instrument_family_str')
    if fam: agrupamento_notas.setdefault(fam, []).append(nota_id)

matriz_de_tarefas = []
contador_par = 0

for fam, notas in agrupamento_notas.items():
    if not dicionario_compatibilidade.get(fam): continue
    amostra = random.sample(notas, min(NOTAS_POR_FAMILIA, len(notas)))
    
    for nota_id in amostra:
        # O RANDOM.SAMPLE RESTAURADO! Controlando a explosão de pares por nota.
        pares_disponiveis = dicionario_compatibilidade[fam]
        pares_escolhidos = random.sample(pares_disponiveis, min(PARES_POR_NOTA, len(pares_disponiveis)))
        
        for par in pares_escolhidos:
            ancora, oposto = par[0], par[1]
            info_a, info_o = vocabulario_dsp[ancora], vocabulario_dsp[oposto]
            id_par = f"par_{contador_par:05d}"
            contador_par += 1
            
            matriz_de_tarefas.append({
                "nota_fonte": nota_id, "familia_instrumento": fam, "grupo_acustico": f"validado_{fam}",
                "descritor": ancora, "tipo_efeito": info_a['tipo'], "params_dsp": info_a['params'],
                "consistencia": info_a['consistencia'], "eh_negativo_dificil": False,
                "eh_multipalavra": info_a['eh_multipalavra'], "id_par_hard_negative": id_par, "oposto_de": None
            })
            matriz_de_tarefas.append({
                "nota_fonte": nota_id, "familia_instrumento": fam, "grupo_acustico": f"validado_{fam}",
                "descritor": oposto, "tipo_efeito": info_o['tipo'], "params_dsp": info_o['params'],
                "consistencia": info_o['consistencia'], "eh_negativo_dificil": True,
                "eh_multipalavra": info_o['eh_multipalavra'], "id_par_hard_negative": id_par, "oposto_de": ancora
            })

df_blueprint = pd.DataFrame(matriz_de_tarefas)

# --- 5. Relatório e Exportação Parquet ---
DIR_OUTPUT = "dataset_fase1_blueprints_valid"
os.makedirs(DIR_OUTPUT, exist_ok=True)
df_blueprint.to_parquet(os.path.join(DIR_OUTPUT, "blueprint_fase1.parquet"), index=False)

df_eq_ouro_export = pd.DataFrame([{'text': k, **v} for k, v in vocabulario_dsp.items() if v['tipo'] == 'eq'])
df_reverb_ouro_export = pd.DataFrame([{'text': k, **v} for k, v in vocabulario_dsp.items() if v['tipo'] == 'reverb'])

if not df_eq_ouro_export.empty: df_eq_ouro_export.to_parquet(os.path.join(DIR_OUTPUT, "vocabulario_eq_ouro.parquet"), index=False)
if not df_reverb_ouro_export.empty: df_reverb_ouro_export.to_parquet(os.path.join(DIR_OUTPUT, "vocabulario_reverb_ouro.parquet"), index=False)

print("\n" + "=" * 70)
print("  BLUEPRINT FASE 1 — RELATÓRIO FINAL")
print("=" * 70)
print(f"  Entradas na matriz de tarefas:         {len(df_blueprint):>8}")
print(f"  Dataset estimado (×5 intensidades):    {len(df_blueprint) * 5:>8} amostras")

print("\n  Sobrevivência Pós-Trava de Energia (Pico > 0.25):")
total_pares_possiveis = len(PARES_OPOSTOS_OURO)
for fam, pares in dicionario_compatibilidade.items():
    if len(pares) > 0:
        print(f"  -> [{fam.upper():<10}]: Aprovou {len(pares)}/{total_pares_possiveis} pares.")
    else:
        print(f"  -> [{fam.upper():<10}]: REJEITADO (Sem energia espectral suficiente).")

print(f"\n  Arquivos exportados para a pasta '{DIR_OUTPUT}/':")
print("    -> blueprint_fase1.parquet")
print("    -> vocabulario_eq_ouro.parquet")
print("    -> vocabulario_reverb_ouro.parquet")

 FASE 1: CURADORIA SOCIALFX (EQ + REVERB) E MAPEAMENTO EMPÍRICO
Baixando SocialFX (EQ e Reverb) do Hugging Face...

Analisando distribuição de energia harmônica no NSynth Valid...


Varrendo Espectros: 100%|██████████████████████████████████████████████████████| 12678/12678 [00:07<00:00, 1703.56it/s]


Construindo Dicionário Dinâmico de Compatibilidade (Corte por Pico: 0.25)...

Gerando Blueprint Otimizado por Compatibilidade...

  BLUEPRINT FASE 1 — RELATÓRIO FINAL
  Entradas na matriz de tarefas:             1920
  Dataset estimado (×5 intensidades):        9600 amostras

  Sobrevivência Pós-Trava de Energia (Pico > 0.25):
  -> [KEYBOARD  ]: Aprovou 5/18 pares.
  -> [BASS      ]: Aprovou 5/18 pares.
  -> [MALLET    ]: REJEITADO (Sem energia espectral suficiente).
  -> [ORGAN     ]: Aprovou 5/18 pares.
  -> [BRASS     ]: Aprovou 5/18 pares.
  -> [GUITAR    ]: Aprovou 5/18 pares.
  -> [STRING    ]: Aprovou 5/18 pares.
  -> [REED      ]: Aprovou 5/18 pares.
  -> [VOCAL     ]: Aprovou 5/18 pares.
  -> [FLUTE     ]: REJEITADO (Sem energia espectral suficiente).

  Arquivos exportados para a pasta 'dataset_fase1_blueprints_valid/':
    -> blueprint_fase1.parquet
    -> vocabulario_eq_ouro.parquet
    -> vocabulario_reverb_ouro.parquet


In [12]:
# =============================================================================
# FASE 1: CURADORIA SOCIALFX (CURVAS DE OURO) E MAPEAMENTO ACÚSTICO (PICO 0.25)
# =============================================================================
import os
import ast
import json
import random
import certifi
import numpy as np
import pandas as pd
import librosa
from datasets import load_dataset
from tqdm import tqdm

# Resiliência de I/O: Contorno de políticas estritas de SSL e desativação 
# de motores experimentais do HuggingFace (Xet) para prevenir falhas de download 
# em servidores de processamento sem terminal gráfico.
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("Iniciando Fase 1 (Consolidada): Curadoria, Curvas de Ouro e Hard Negatives")
print("=" * 70)


# =============================================================================
# BLOCO 1 — PARÂMETROS GLOBAIS
# =============================================================================

# --- Qualidade do EQ ---
LIMIAR_CONSISTENCIA_EQ = 0.500
# JUSTIFICATIVA CIENTÍFICA: A análise topológica do SocialFX revela que descritores 
# de timbre espectral (bright, dark) ultrapassam assintoticamente 0.80. A adoção 
# de um limiar mais permissivo incorpora termos com alta precisão descritiva localizados 
# na cauda longa da distribuição, maximizando a variância lexical de treino. A 
# desambiguação subsequente é garantida pela ortogonalidade introduzida nos Hard Negatives.

# --- Qualidade do Reverb ---
MIN_PALAVRAS_AGREED_REVERB = 1
# JUSTIFICATIVA CIENTÍFICA: A força de consenso para reverberação baseia-se na contagem 
# absoluta de palavras válidas retidas pelo crowdsourcing. Exigir >= 1 palavra válida 
# no atributo `agreed` expurga respostas sentinelas e amostras de descarte oriundas dos Turkers.

MAX_VIRGULAS_TEXT_REVERB = 2
# JUSTIFICATIVA CIENTÍFICA: Modelos Joint Embeddings (LAION-CLAP) empregam codificadores 
# de texto subword-based (RoBERTa). Rótulos multi-palavras densamente concatenados degradam 
# os mapas de atenção bidirecional, induzindo estados OOV latentes. O limite rígido de 
# 2 vírgulas consolida compostos sintaticamente processáveis.

TOP_K_POR_SPLIT = 300
# Estabelece um teto de expansão do vocabulário, estabilizando o uso da memória de 
# projeção textual no pipeline de treinamento contrastivo.

# --- Escala do NSynth ---
NOTAS_POR_FAMILIA = 60
# JUSTIFICATIVA CIENTÍFICA: Extração de amostras controladas por família 
# instrumental evita viés associativo condicional no preditor acústico.

PARES_POR_NOTA = 2
# A geração de 2 pares por nota dimensiona o lote total, garantindo volume 
# estatístico adequado após a propagação de 5 intensidades na etapa DSP.

freqs_eq = np.logspace(np.log10(20), np.log10(20000), num=40)

# Dicionário Expandido de Pares Contrastivos (Seus Originais + Novos de Espaço)
PARES_OPOSTOS_OURO = {
    "bright": "dark", "dark": "bright",
    "warm": "tinny", "tinny": "warm",
    "muffled": "crisp", "crisp": "muffled",
    "harsh": "soft", "soft": "harsh",
    "metallic": "mellow", "mellow": "metallic",
    "deep": "thin", "thin": "deep",
    "bassy": "trebly", "trebly": "bassy",
    "spacious": "dry", "dry": "spacious",
    "distant": "close", "close": "distant"
}


# =============================================================================
# BLOCO 2 E 3 — FUNÇÕES AUXILIARES
# =============================================================================

def parse_extra_data(extra_str):
    """Extração sintaticamente segura de dicionários aninhados em formato string."""
    if isinstance(extra_str, str):
        try: return ast.literal_eval(extra_str)
        except (ValueError, SyntaxError): return {}
    return extra_str if isinstance(extra_str, dict) else {}

def parse_params(param_val):
    """Conversão de metadados paramétricos de áudio contidos em strings iteráveis."""
    if isinstance(param_val, str):
        try: return ast.literal_eval(param_val)
        except (ValueError, SyntaxError): return []
    return param_val if param_val is not None else []

def calc_consensus_strength_reverb(extra_dict):
    """Avalia a densidade de respostas expurgando artefatos de ausência de escolha."""
    agreed_raw = extra_dict.get('agreed', '')
    if not agreed_raw: return 0
    tokens_validos = [
        w.strip() for w in agreed_raw.split(',')
        if w.strip() and w.strip().lower() not in ('none', 'none of the above', '')
    ]
    return len(tokens_validos)


# =============================================================================
# BLOCO 4 — EXTRAÇÃO E FILTRAGEM DO SocialFX
# =============================================================================

print("\nBaixando SocialFX do Hugging Face...")
dataset_eq     = load_dataset("seungheondoh/socialfx-original", split="eq").to_pandas()
dataset_reverb = load_dataset("seungheondoh/socialfx-original", split="reverb").to_pandas()

# --- EQ ---
dataset_eq['extra_dict'] = dataset_eq['extra'].apply(parse_extra_data)
dataset_eq['lang']       = dataset_eq['extra_dict'].apply(lambda x: x.get('lang', ''))
dataset_eq['consistency'] = dataset_eq['extra_dict'].apply(lambda x: float(x.get('ratings_consistency', 0.0)))

# JUSTIFICATIVA CIENTÍFICA: A blacklist visa estritamente artefatos não-acústicos 
# inseridos no crowdsourcing, limitando viés analítico ao preservar afecções válidas.
BLACKLIST_EQ = ['taco', 'baseball', 'disgusting', 'genius', 'poor']

df_eq_filtrado = dataset_eq[
    (dataset_eq['lang'] == 'English') &
    (dataset_eq['consistency'] >= LIMIAR_CONSISTENCIA_EQ) &
    (~dataset_eq['text'].isin(BLACKLIST_EQ))
].copy()
df_eq_filtrado = df_eq_filtrado.sort_values('consistency', ascending=False).drop_duplicates(subset=['text'], keep='first')
df_eq_filtrado['param_values_list'] = df_eq_filtrado['param_values'].apply(parse_params)
df_eq_filtrado['tipo_efeito']       = 'eq'
df_eq_filtrado['consistencia']      = df_eq_filtrado['consistency']
df_eq_ouro = df_eq_filtrado.head(TOP_K_POR_SPLIT).copy()

print(f"  EQ     -> {len(df_eq_ouro):>4} termos | range consistência: [{df_eq_ouro.iloc[-1]['consistency']:.3f} - {df_eq_ouro.iloc[0]['consistency']:.3f}]")

# --- REVERB ---
dataset_reverb['extra_dict']         = dataset_reverb['extra'].apply(parse_extra_data)
dataset_reverb['lang']               = dataset_reverb['extra_dict'].apply(lambda x: x.get('lang', ''))
dataset_reverb['consensus_strength'] = dataset_reverb['extra_dict'].apply(calc_consensus_strength_reverb)

df_reverb_filtrado = dataset_reverb[
    (dataset_reverb['lang'] == 'English') &
    (dataset_reverb['consensus_strength'] >= MIN_PALAVRAS_AGREED_REVERB) &
    (dataset_reverb['text'].str.count(',') <= MAX_VIRGULAS_TEXT_REVERB)
].copy()

# JUSTIFICATIVA CIENTÍFICA: Transposição de tokens aglutinados para 
# separação ortográfica apropriada. A inserção de espaços assegura a interpretação de 
# múltiplos descritores como entidades semânticas compostas no Text Encoder.
df_reverb_filtrado['text'] = df_reverb_filtrado['text'].str.replace(',', ', ', regex=False)
df_reverb_filtrado = df_reverb_filtrado.sort_values('consensus_strength', ascending=False).drop_duplicates(subset=['text'], keep='first')
df_reverb_filtrado['param_values_list'] = df_reverb_filtrado['param_values'].apply(parse_params)
df_reverb_filtrado['tipo_efeito']       = 'reverb'
df_reverb_filtrado['consistencia']      = df_reverb_filtrado['consensus_strength'].astype(float)
df_reverb_ouro = df_reverb_filtrado.head(TOP_K_POR_SPLIT).copy()

print(f"  Reverb -> {len(df_reverb_ouro):>4} termos | range consensus: [{int(df_reverb_ouro.iloc[-1]['consensus_strength'])} - {int(df_reverb_ouro.iloc[0]['consensus_strength'])} palavras]")


# =============================================================================
# BLOCO 5 — VOCABULÁRIO DSP UNIFICADO (CURVAS DE OURO E POLARIDADE)
# =============================================================================

vocabulario_dsp = {}

# Trava semântica: Impede que o EQ capture palavras que devem obrigatoriamente
# ser tratadas como espacialidade pelo Reverb.
TERMOS_ESTRITAMENTE_ESPACIAIS = {"spacious", "echo", "dry", "distant", "close"}

# =============================================================================
# JUSTIFICATIVA CIENTÍFICA: CORREÇÃO DE POLARIDADE DO SOCIALFX
# A validação estatística por extração de Deltas (Tau de Kendall e Pearson) 
# detectou que o dataset original 'socialfx' possui correlações fortemente 
# negativas para certos descritores (ex: Bright = -0.91). Isso indica que as 
# médias dos ganhos (crowdsourced) estão com a polaridade invertida.
# Aplicamos aqui uma inversão algébrica (multiplicação por -1) para restaurar 
# a coerência física do espectro sem descartar os dados.
# =============================================================================
TERMOS_INVERTIDOS_PELO_SOCIALFX = {"bright", "dark", "tinny", "harsh", "metallic", "reed"}

print("\nCalculando as Curvas de Ouro (Média Ponderada) para os pares...")

# Processamento EQ
for texto, group in df_eq_filtrado.groupby('text'):
    if texto in TERMOS_ESTRITAMENTE_ESPACIAIS: continue
    
    curvas_validas = group[group['param_values_list'].apply(len) == 40]
    if len(curvas_validas) == 0: continue
        
    curvas = np.array(curvas_validas['param_values_list'].tolist())
    pesos = curvas_validas['consistency'].values
    
    # Média Ponderada
    curva_ouro = np.average(curvas, axis=0, weights=pesos) if np.sum(pesos) > 0 else np.mean(curvas, axis=0)
    
    # INTERVENÇÃO FÍSICA: Correção de Polaridade
    if texto in TERMOS_INVERTIDOS_PELO_SOCIALFX:
        curva_ouro = -curva_ouro
        
    vocabulario_dsp[texto] = {
        'tipo': 'eq',
        'params': curva_ouro.tolist(),
        'consistencia': float(np.mean(pesos)),
        'eh_multipalavra': False,
        'bandas_ativas': np.where(np.abs(curva_ouro) > 0.2)[0].tolist()
    }

# Processamento Reverb
for _, row in df_reverb_ouro.iterrows():
    texto = row['text']
    if texto not in vocabulario_dsp:
        vocabulario_dsp[texto] = {
            'tipo': 'reverb',
            'params': row['param_values_list'],
            'consistencia': row['consistencia'],
            'eh_multipalavra': ', ' in texto,
            'bandas_ativas': []
        }


# =============================================================================
# BLOCO 6 — ANÁLISE ESPECTRAL E SELEÇÃO DE NOTAS (TRAVA DINÂMICA)
# =============================================================================

print("\nCarregando notas base do NSynth Valid...")
with open('nsynth-valid/examples.json', 'r') as f:
    nsynth_meta = json.load(f)

# Seleção de Notas
agrupamento_notas = {}
for nota_id, info in nsynth_meta.items():
    fam = info.get('instrument_family_str')
    if fam: agrupamento_notas.setdefault(fam, []).append(nota_id)

notas_base_selecionadas = {}
total_notas_secas = 0
for fam, notas in agrupamento_notas.items():
    if notas:
        amostra = random.sample(notas, min(NOTAS_POR_FAMILIA, len(notas)))
        notas_base_selecionadas[fam] = amostra
        total_notas_secas += len(amostra)

print(f"  → Total de notas selecionadas de forma dinâmica: {total_notas_secas}")

# O Raio-X Dinâmico com Librosa
print("Analisando distribuição de energia harmônica nas notas selecionadas...")
energia_instrumentos = {}
contagem_instrumentos = {}
DIR_AUDIO_DRY = "nsynth-valid/audio"

for fam, notas in notas_base_selecionadas.items():
    energia_instrumentos[fam] = np.zeros(40)
    contagem_instrumentos[fam] = 0
    
    for nota_id in tqdm(notas, desc=f"Varrendo {fam.upper()}", leave=False):
        caminho = os.path.join(DIR_AUDIO_DRY, f"{nota_id}.wav")
        if not os.path.exists(caminho): continue
            
        try:
            y, sr = librosa.load(caminho, sr=22050, mono=True)
            stft = np.abs(librosa.stft(y))
            fft_freqs = librosa.fft_frequencies(sr=sr)
            espectro_medio = np.mean(stft, axis=1)
            
            energia_40_bandas = np.zeros(40)
            for i, f_alvo in enumerate(freqs_eq):
                idx = np.where((fft_freqs >= f_alvo * 0.8) & (fft_freqs <= f_alvo * 1.2))[0]
                if len(idx) > 0: energia_40_bandas[i] = np.mean(espectro_medio[idx])
                    
            if np.max(energia_40_bandas) > 0: energia_40_bandas /= np.max(energia_40_bandas)
            energia_instrumentos[fam] += energia_40_bandas
            contagem_instrumentos[fam] += 1
        except: continue
            
    if contagem_instrumentos[fam] > 0: 
        energia_instrumentos[fam] /= contagem_instrumentos[fam]

# Substitui o JSON antigo pelo dicionário dinâmico baseado na Trava de Pico (0.25)
print("\nConstruindo Dicionário Dinâmico de Compatibilidade (Corte por Pico: 0.25)...")
MAPEAMENTO_EMPIRICO = {fam: [] for fam in energia_instrumentos.keys()}

for fam, energia_perfil in energia_instrumentos.items():
    for termo, oposto in PARES_OPOSTOS_OURO.items():
        if termo not in vocabulario_dsp or oposto not in vocabulario_dsp: continue
            
        info_a = vocabulario_dsp[termo]
        info_o = vocabulario_dsp[oposto]
        par_ordenado = sorted([termo, oposto])
        
        if par_ordenado in MAPEAMENTO_EMPIRICO[fam]: continue
            
        if info_a['tipo'] == 'eq' and info_o['tipo'] == 'eq':
            bandas = set(info_a['bandas_ativas'] + info_o['bandas_ativas'])
            if not bandas: continue
            
            # Trava Cirúrgica de Pico
            energia_pico_alvo = np.max([energia_perfil[b] for b in bandas])
            if energia_pico_alvo > 0.25: 
                MAPEAMENTO_EMPIRICO[fam].append(par_ordenado)
        else:
            MAPEAMENTO_EMPIRICO[fam].append(par_ordenado)


# =============================================================================
# BLOCO 7 — GERAÇÃO DO BLUEPRINT (Matriz de Tarefas)
# =============================================================================

print("\nGerando Blueprint Otimizado por Compatibilidade de Energia...")
matriz_de_tarefas = []
contador_par = 0

for fam, pares_compativeis in MAPEAMENTO_EMPIRICO.items():
    if fam not in notas_base_selecionadas:
        continue
        
    notas_da_familia = notas_base_selecionadas[fam]
    
    for nota_id in notas_da_familia:
        if not pares_compativeis: 
            continue  
            
        # Seleciona os pares baseando-se na sua constante PARES_POR_NOTA
        pares_escolhidos = random.sample(pares_compativeis, min(PARES_POR_NOTA, len(pares_compativeis)))
        
        for par in pares_escolhidos:
            palavra_ancora = par[0]
            oposto = par[1]
            
            info_a = vocabulario_dsp[palavra_ancora]
            id_par = f"par_{contador_par:05d}"
            contador_par += 1
            
            # Ingestão da Âncora (Par Positivo)
            matriz_de_tarefas.append({
                "nota_fonte":           nota_id,
                "familia_instrumento":  fam,
                "grupo_acustico":       f"validado_{fam}",
                "descritor":            palavra_ancora,
                "tipo_efeito":          info_a['tipo'],
                "params_dsp":           info_a['params'],
                "consistencia":         info_a['consistencia'],
                "eh_negativo_dificil":  False,
                "eh_multipalavra":      info_a['eh_multipalavra'],
                "id_par_hard_negative": id_par,
                "oposto_de":            None,
            })
            
            # Ingestão do Hard Negative
            info_o = vocabulario_dsp[oposto]
            matriz_de_tarefas.append({
                "nota_fonte":           nota_id,
                "familia_instrumento":  fam,
                "grupo_acustico":       f"validado_{fam}",
                "descritor":            oposto,
                "tipo_efeito":          info_o['tipo'],
                "params_dsp":           info_o['params'],
                "consistencia":         info_o['consistencia'],
                "eh_negativo_dificil":  True,
                "eh_multipalavra":      info_o['eh_multipalavra'],
                "id_par_hard_negative": id_par, 
                "oposto_de":            palavra_ancora,
            })

df_blueprint = pd.DataFrame(matriz_de_tarefas)
print(f"✓ Novo Blueprint estruturado: {len(df_blueprint)} tarefas contrastivas puras geradas!")


# =============================================================================
# BLOCO 8 — RELATÓRIO E EXPORTAÇÃO
# =============================================================================

total_estimado = len(df_blueprint) * 5

print("\n" + "=" * 70)
print("  BLUEPRINT FASE 1 — RELATÓRIO FINAL")
print("=" * 70)
print(f"\n  Notas fonte selecionadas:              {total_notas_secas:>8}")
print(f"  Entradas na matriz de tarefas:         {len(df_blueprint):>8}")
print(f"  └─ Âncoras (pares positivos):          {df_blueprint[~df_blueprint['eh_negativo_dificil']].shape[0]:>8}")
print(f"  └─ Hard Negatives estruturais:         {df_blueprint[df_blueprint['eh_negativo_dificil']].shape[0]:>8}")
print(f"  └─ Âncoras multi-palavra (reverb):     {df_blueprint[df_blueprint['eh_multipalavra']].shape[0]:>8}")
print(f"\n  Dataset estimado (×5 intensidades):    {total_estimado:>8} amostras")

print("\n  Sobrevivência Pós-Trava de Energia (Pico > 0.25):")
total_pares_possiveis = len(PARES_OPOSTOS_OURO)
for fam, pares in MAPEAMENTO_EMPIRICO.items():
    if len(pares) > 0:
        print(f"  -> [{fam.upper():<10}]: Aprovou {len(pares)}/{total_pares_possiveis} pares.")
    else:
        print(f"  -> [{fam.upper():<10}]: REJEITADO (Sem energia espectral suficiente).")

print("\n  Distribuição por tipo de efeito DSP:")
print(df_blueprint.groupby(['tipo_efeito', 'eh_negativo_dificil']).size().rename("contagem").to_string())

print("\n  Distribuição por grupo acústico:")
print(df_blueprint.groupby('grupo_acustico').size().rename("contagem").to_string())

print("\n  Pares de Hard Negatives (amostra de validação):")
pares_demo = (
    df_blueprint[df_blueprint['id_par_hard_negative'].duplicated(keep=False)]
   .sort_values(['id_par_hard_negative', 'eh_negativo_dificil'])
   .head(8)[['nota_fonte', 'descritor', 'tipo_efeito', 'eh_negativo_dificil', 'id_par_hard_negative', 'consistencia']]
   .to_string(index=False)
)
print(pares_demo)

DIR_OUTPUT = "dataset_fase1_blueprints_valid"
os.makedirs(DIR_OUTPUT, exist_ok=True)

path_blueprint = os.path.join(DIR_OUTPUT, "blueprint_fase1.parquet")
path_eq = os.path.join(DIR_OUTPUT, "vocabulario_eq_ouro.parquet")
path_reverb = os.path.join(DIR_OUTPUT, "vocabulario_reverb_ouro.parquet")

df_blueprint.to_parquet(path_blueprint, index=False)
df_eq_ouro.to_parquet(path_eq, index=False)
df_reverb_ouro.to_parquet(path_reverb, index=False)

print(f"\n  Arquivos exportados para a pasta '{DIR_OUTPUT}/':")
print("    -> blueprint_fase1.parquet")
print("    -> vocabulario_eq_ouro.parquet")
print("    -> vocabulario_reverb_ouro.parquet")
print("=" * 70)

Iniciando Fase 1 (Consolidada): Curadoria, Curvas de Ouro e Hard Negatives

Baixando SocialFX do Hugging Face...
  EQ     ->  207 termos | range consistência: [0.501 - 0.931]
  Reverb ->  300 termos | range consensus: [5 - 11 palavras]

Calculando as Curvas de Ouro (Média Ponderada) para os pares...

Carregando notas base do NSynth Valid...
  → Total de notas selecionadas de forma dinâmica: 600
Analisando distribuição de energia harmônica nas notas selecionadas...



Construindo Dicionário Dinâmico de Compatibilidade (Corte por Pico: 0.25)...

Gerando Blueprint Otimizado por Compatibilidade de Energia...
✓ Novo Blueprint estruturado: 1920 tarefas contrastivas puras geradas!

  BLUEPRINT FASE 1 — RELATÓRIO FINAL

  Notas fonte selecionadas:                   600
  Entradas na matriz de tarefas:             1920
  └─ Âncoras (pares positivos):               960
  └─ Hard Negatives estruturais:              960
  └─ Âncoras multi-palavra (reverb):            0

  Dataset estimado (×5 intensidades):        9600 amostras

  Sobrevivência Pós-Trava de Energia (Pico > 0.25):
  -> [KEYBOARD  ]: Aprovou 5/18 pares.
  -> [BASS      ]: Aprovou 5/18 pares.
  -> [MALLET    ]: REJEITADO (Sem energia espectral suficiente).
  -> [ORGAN     ]: Aprovou 5/18 pares.
  -> [BRASS     ]: Aprovou 5/18 pares.
  -> [GUITAR    ]: Aprovou 5/18 pares.
  -> [STRING    ]: Aprovou 5/18 pares.
  -> [REED      ]: Aprovou 5/18 pares.
  -> [VOCAL     ]: Aprovou 5/18 pares.
  -> [FLU

In [14]:
import warnings
warnings.filterwarnings("ignore", module="pyloudnorm")

In [15]:
# =============================================================================
# FASE 2: MOTOR DSP UNIFICADO E GERAÇÃO MATRICIAL
# =============================================================================
import os
import gc
import math
import random
import numpy as np
import pandas as pd
import soundfile as sf
import pyloudnorm as pyln
from scipy.signal import resample_poly
from tqdm import tqdm
from pedalboard import Pedalboard, PeakFilter, Reverb

print("Iniciando Fase 2: Motor DSP Unificado e Geração Matricial")
print("=" * 80)

# =============================================================================
# BLOCO 1 — CONFIGURAÇÕES DE DIRETÓRIO E ESCALA
# =============================================================================

# Diretórios de I/O
DIR_BLUEPRINTS = "dataset_fase1_blueprints_valid"
DIR_AUDIO_IN = "nsynth-valid/audio"
DIR_AUDIO_OUT = "dataset_fase2_valid_elite/audio"
DIR_META_OUT = "dataset_fase2_valid_elite"

os.makedirs(DIR_AUDIO_OUT, exist_ok=True)

# Escala de intensidades e seus respectivos modificadores textuais.
# JUSTIFICATIVA CIENTÍFICA: A injeção de advérbios correlacionados à magnitude
# do processamento DSP permite que o Joint Language Embedding (CLAP) aprenda 
# a topologia contínua da intensidade do efeito, não apenas a classe binária.
INTENSIDADES = [0.2, 0.4, 0.6, 0.8, 1.0]

MODIFICADORES = {
    0.2: ["slightly", "faintly", "a hint of"],
    0.4: ["mildly", "a bit", "marginally"],
    0.6: ["moderately", "somewhat", "partially"],
    0.8: ["mostly", "substantially", "largely"],
    1.0: ["", "", ""] # Intensidade máxima usa o adjetivo puro (ex: "a bright guitar")
}


# =============================================================================
# BLOCO 2 — MOTOR DSP UNIFICADO (COM NORMALIZAÇÃO LUFS)
# =============================================================================

# Vetor de 40 frequências logarítmicas (padrão do SocialFX)
freqs_eq = np.logspace(np.log10(20), np.log10(20000), num=40)
ALVO_LUFS = -23.0  # Padrão da indústria (EBU R128) para headroom seguro

def processar_dsp_monotonico(audio_data, sample_rate, tipo_efeito, parametros, intensidade):
    """
    Roteia o processamento baseado no tipo de efeito (EQ ou Reverb)
    aplicando escalonamento paramétrico linear, normalização de ganho (LUFS) e proteção Anti-Clipping.
    Inclui trava dinâmica baseada no limite de Nyquist para evitar aliasing.
    """
    if tipo_efeito == 'eq':
        filtros = []
        
        # 1. Encontra o pico máximo desenhado pelo humano (seja corte ou ganho)
        max_ganho = np.max(np.abs(parametros)) if len(parametros) > 0 else 1.0
        
        # 2. Fator de normalização: Forçamos o pico do EQ para 15 dB (efeito dramático)
        fator_amplificacao = 15.0 / max_ganho if max_ganho > 0 else 1.0

        # JUSTIFICATIVA CIENTÍFICA: Alinhamento com o Teorema de Nyquist-Shannon. A imposição 
        # de uma trava de segurança dinâmica baseada em Fs/2 impede a extrapolação do teto 
        # de representação digital do sinal. Isso mitiga a introdução de artefatos de aliasing, 
        # assegurando que o espaço de embeddings do CLAP seja treinado com variações tímbricas 
        # puras e livres de distorções espelhadas no espectro audível.
        limite_nyquist = sample_rate / 2.0

        for i, freq in enumerate(freqs_eq):
            # Adicionada a trava de Nyquist com margem de segurança ao laço existente
            if i < len(parametros) and freq < (limite_nyquist * 0.95):
                # 3. Amplifica a curva e depois corta na proporção da intensidade (0.2 a 1.0)
                ganho_escalado = (parametros[i] * fator_amplificacao) * intensidade
                
                # Ignora filtros quase inaudíveis para poupar processamento (limiar ajustado para 0.5)
                if abs(ganho_escalado) > 0.5: 
                    filtros.append(PeakFilter(cutoff_frequency_hz=freq, gain_db=ganho_escalado, q=1.5))
        
        board = Pedalboard(filtros)
        
    elif tipo_efeito == 'reverb':
        # Reverb mais agressivo para garantir clusterização nas métricas
        room_size_calc = 0.2 + (intensidade * 0.8)  # De 0.2 a 1.0 (caudas enormes no máximo)
        wet_level_calc = intensidade * 0.9          # Deixa o som quase 100% "molhado" no máximo
        board = Pedalboard([
            Reverb(room_size=room_size_calc, wet_level=wet_level_calc, dry_level=1.0)
        ])
        
    else:
        raise ValueError(f"Tipo de efeito DSP desconhecido: {tipo_efeito}")

    # Conversão de segurança para Mono forçada (crucial para o Motor DSP)
    if len(audio_data.shape) > 1:
        audio_data = audio_data[:, 0]

    # Processamento do sinal
    audio_processado = board(audio_data, sample_rate)

    # =========================================================================
    # JUSTIFICATIVA CIENTÍFICA: NORMALIZAÇÃO PERCEPTIVA (LUFS)
    # A normalização de Loudness assegura que todos os timbres gerados possuam 
    # o mesmo volume perceptivo. Isso anula o viés de amplitude, impedindo que  
    # o modelo contrastivo (CLAP) crie correlações espúrias (ex: "bright = alto"). 
    # O headroom de -23 LUFS garante margem segura contra o digital clipping.
    # =========================================================================
    try:
        meter = pyln.Meter(sample_rate) # Cria o medidor acústico
        loudness_atual = meter.integrated_loudness(audio_processado)
        
        # Normaliza o áudio gerado para o alvo (-23 LUFS)
        audio_processado = pyln.normalize.loudness(audio_processado, loudness_atual, ALVO_LUFS)
        
        # Trava de segurança digital secundária (caso um transitório escape)
        pico_maximo = np.max(np.abs(audio_processado))
        if pico_maximo > 1.0:
            audio_processado = audio_processado / pico_maximo
            
    except ValueError:
        # Fallback de segurança caso o DSP retorne silêncio absoluto
        pass

    return audio_processado

# =============================================================================
# BLOCO 3 — GERADOR DE PROMPTS SEMÂNTICOS
# =============================================================================

def gerar_prompt_semantico(palavra, familia_inst, intensidade, tipo_efeito):
    """
    Constrói a representação textual ancorada combinando o modificador de 
    magnitude, o descritor acústico e a fonte sonora.
    """
    modificador = random.choice(MODIFICADORES[intensidade])
    prefixo = f"{modificador} {palavra}".strip()
    contexto = familia_inst.replace('_', ' ') # ex: "synth lead"
    
    if tipo_efeito == 'eq':
        templates = [
            f"a {prefixo} {contexto}",
            f"{contexto} with a {prefixo} equalization",
            f"a recording of a {contexto} modified to sound {prefixo}"
        ]
    elif tipo_efeito == 'reverb':
        templates = [
            f"a {contexto} with {prefixo} reverberation",
            f"a {contexto} placed in a {prefixo} acoustic space",
            f"{contexto} processed to have a {prefixo} spatial effect"
        ]
    return random.choice(templates)

# =============================================================================
# BLOCO 4 — LAÇO DE PROCESSAMENTO (VIA SOUNDFILE + SCIPY UPSAMPLING)
# =============================================================================

print("Carregando Blueprint da Fase 1...")
try:
   df_blueprint = pd.read_parquet(os.path.join(DIR_BLUEPRINTS, "blueprint_fase1.parquet"))
except FileNotFoundError:
    raise RuntimeError("Blueprint não encontrado. Execute a Fase 1 primeiro.")

dataset_metadata = []
erros_leitura = 0

# Dicionário de cache inteligente na RAM: Evita recalcular o upsampling da mesma nota
cache_upsampled = {}

print(f"Iniciando renderização de {len(df_blueprint) * len(INTENSIDADES)} amostras de áudio...")

# Iteração linha a linha pelo Blueprint
for index, row in tqdm(df_blueprint.iterrows(), total=len(df_blueprint), desc="Processando DSP"):
    nota_id = row['nota_fonte']
    descritor = row['descritor']
    tipo_efeito = row['tipo_efeito']
    parametros = row['params_dsp']
    familia = row['familia_instrumento']
    
    audio_path = os.path.join(DIR_AUDIO_IN, f"{nota_id}.wav")
    
    # =========================================================================
    # 1. CARREGAMENTO E UPSAMPLING DE ALTA FIDELIDADE
    # =========================================================================
    # Se a nota ainda não foi processada e salva na RAM, fazemos o upsampling
    if nota_id not in cache_upsampled:
        try:
            # Leitura nativa e segura do arquivo
            audio_data, sr_original = sf.read(audio_path)
            
            # Força Mono por segurança
            if len(audio_data.shape) > 1:
                audio_data = audio_data[:, 0]
                
            # JUSTIFICATIVA CIENTÍFICA: Reamostragem (Upsampling) de Alta Fidelidade. A conversão 
            # do substrato acústico original do NSynth (16 kHz) para 44.1 kHz via interpolação de 
            # banda limitada expande o headroom digital e eleva o limite de Nyquist para 22.05 kHz. 
            # Esse procedimento viabiliza a aplicação integral dos descritores espectrais de alta 
            # frequência do SocialFX sem causar distorções espelhadas.
            if sr_original != 44100:
                # Cálculo exato da proporção para o filtro polyphase do Scipy (Substituto robusto ao Librosa)
                fator_comum = math.gcd(44100, sr_original)
                up = 44100 // fator_comum
                down = sr_original // fator_comum
                audio_data = resample_poly(audio_data, up, down)
                
            cache_upsampled[nota_id] = audio_data
        except Exception:
            erros_leitura += 1
            continue
            
    # Recupera o áudio original interpolado da RAM (instantâneo)
    audio_data = cache_upsampled[nota_id]
    sr = 44100
    
    # =========================================================================
    # 2. LAÇO DE INTENSIDADES (GERAÇÃO DE DADOS)
    # =========================================================================
    for i in INTENSIDADES:
        pct = int(i * 100)
        
        # 1. Aplica o processamento matemático (Protegido pela trava de Nyquist e normalização LUFS no Bloco 2)
        audio_processado = processar_dsp_monotonico(
            audio_data=audio_data, 
            sample_rate=sr, 
            tipo_efeito=tipo_efeito, 
            parametros=parametros, 
            intensidade=i
        )
        
        # 2. Salva o arquivo em disco usando o Soundfile (Nativo e seguro)
        nome_arquivo = f"{tipo_efeito}_{descritor.replace(' ', '_')}_{pct}pct_{nota_id}.wav"
        caminho_salvo = os.path.join(DIR_AUDIO_OUT, nome_arquivo)
        sf.write(caminho_salvo, audio_processado, sr)
        
        # 3. Gera a legenda contrastiva (Prompt)
        prompt = gerar_prompt_semantico(descritor, familia, i, tipo_efeito)
        
        # 4. Registra no metadado final
        dataset_metadata.append({
            "file_name": nome_arquivo,
            "text": prompt,
            "keyword": descritor,
            "effect_type": tipo_efeito,
            "intensity": i,
            "original_note": nota_id,
            "is_hard_negative": row['eh_negativo_dificil'],
            "hard_negative_pair_id": row['id_par_hard_negative'],
            "consistency_score": row['consistencia']
        })
        
    # Boas práticas: Coleta de lixo espaçada para manter o ambiente saudável
    if index % 200 == 0:
        gc.collect()

# Exportação do metadado final
df_treino_final = pd.DataFrame(dataset_metadata)
df_treino_final.to_csv(os.path.join(DIR_META_OUT, "metadata.csv"), index=False)

# Limpeza final
cache_upsampled.clear()
gc.collect()

print("\n" + "=" * 80)
print("  RENDERIZAÇÃO CONCLUÍDA")
print("=" * 80)
print(f"  Total de arquivos de áudio gerados: {len(df_treino_final)}")
if erros_leitura > 0:
    print(f"  Aviso: {erros_leitura} notas do NSynth não foram encontradas no diretório fonte.")
print(f"  Metadados de treinamento salvos em: {os.path.join(DIR_META_OUT, 'metadata.csv')}")
print("  Dataset finalizado e pronto para ingestão no modelo CLAP.")

Iniciando Fase 2: Motor DSP Unificado e Geração Matricial
Carregando Blueprint da Fase 1...
Iniciando renderização de 9600 amostras de áudio...


Processando DSP: 100%|█████████████████████████████████████████████████████████████| 1920/1920 [05:41<00:00,  5.61it/s]


  RENDERIZAÇÃO CONCLUÍDA
  Total de arquivos de áudio gerados: 9600
  Metadados de treinamento salvos em: dataset_fase2_valid_elite\metadata.csv
  Dataset finalizado e pronto para ingestão no modelo CLAP.
